# Amplification Must Be Paid: Public Lab

This notebook is not a gallery. It is a ledger worksheet for the proposed Third Law:

> amplification must be paid.

The computation asks one question several ways: when the audit deliberately creates apparent amplification, does it become a free route, or does it land in a named payment channel?

## How to Read This Notebook

Each evidence block follows the same pattern:

1. load the cached CSV rows;
2. state the calculation being performed;
3. recompute the decisive quantity in public view;
4. state what would break the claim.

The helper module is not a theorem oracle. It is just a small public library that loads cached artifacts, recomputes ledger columns, and draws simple figures. The tables below are the important part.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
HELPER = ROOT / "notebooks" / "amplification_lab.py"
sys.path.insert(0, str(ROOT / "notebooks"))

import amplification_lab as lab

MODE = "cached"  # synthetic is only a no-data smoke test, not JHTDB evidence
print(f"public root: {ROOT}")
print(f"helper imported from: {HELPER}")
print(f"mode: {MODE}")

## Artifact Map

The notebook uses cached summary artifacts, not the full raw JHTDB cache. The first check is whether every expected artifact is present. The artifact hashes and row counts are recorded in `data/ARTIFACTS.md`.

In [ ]:
inventory = lab.artifact_table()
display(inventory[["demo", "kind", "path", "exists", "bytes"]])

if MODE == "cached":
    lab.require_cached_data()
    display(Markdown("Cached evidence CSVs are present. The notebook will use real cached summary rows."))
else:
    display(Markdown("Synthetic mode is active. These are schema-compatible toy rows, not JHTDB evidence."))

## Ledger Routing Rule

The paper's public claim is not that every row is already closed. The claim is stricter and easier to falsify: no pressure source gets to disappear. It must be paid, charged, quotient-identified, or left as a named unresolved row.

In [ ]:
display(lab.routing_board())

# Evidence 1: Same-Parent Quotienting

**Attack.** Split a same-parent source into many labels. This inflates `pre_quotient_tubes`.

**Ledger question.** After quotienting, do those labels survive as nonredundant physical packing?

**Calculation.** Compute `created_labels = pre_quotient_tubes - post_quotient_tubes`, then check whether split rows keep positive `R_pack_star` after quotienting.

In [ ]:
same = lab.same_parent_ledger(MODE)
print("Columns used for this calculation:")
print(list(same.columns))

display(lab.same_parent_explanation(MODE))
fig = lab.plot_same_parent(MODE)
plt.show()

display(same.sort_values(["case", "radius_dx", "M"]).head(18))

break_rows = same[same["free_amplification_survives"]]
if break_rows.empty:
    display(Markdown("**Result:** no same-parent split row survives as free post-quotient packing in this cache."))
else:
    display(Markdown("**Potential break:** a split row survived quotienting."))
    display(break_rows)

**How to break Evidence 1.** Find a row with `M > 1`, sustained `post_quotient_tubes`, and positive `R_pack_star` after quotienting. That would mean label amplification survived as physical packing.

# Evidence 2: Renewal-Cascade Jitter

**Attack.** Keep source capture high while perturbing cross-parent packets.

**Ledger question.** Can the perturbation avoid final physical charge?

**Calculation.** On stressed rows (`M > 1`), compute whether high `eta_post` coexists with vanishing `Fphys_star_available` and positive residual packing. That is the free-jitter candidate.

In [ ]:
renewal = lab.renewal_ledger(MODE)
print("Columns used for this calculation:")
print(list(renewal.columns))

display(lab.renewal_explanation(MODE))
fig = lab.plot_renewal(MODE)
plt.show()

view_cols = [
    "case", "ablation_mode", "M", "eta_post", "post_growth_fraction",
    "Fphys_star_available", "C0theta_proxy", "free_jitter_candidate"
]
display(renewal[view_cols].head(20))

break_rows = renewal[renewal["free_jitter_candidate"]]
if break_rows.empty:
    display(Markdown("**Result:** high-capture jitter rows still pay physical charge in this cache."))
else:
    display(Markdown("**Potential break:** high-capture jitter avoided physical charge."))
    display(break_rows)

**How to break Evidence 2.** Produce high `eta_post` under cross-parent jitter with `Fphys_star_available -> 0`, no quotient redundancy, and no named renewal/separation/oscillation charge.

# Evidence 3A: Tail Denominator Ladder

**Attack.** Push pressure into annular tail reservoirs.

**Ledger question.** Is the tail hidden, or is the best available denominator row named?

**Calculation.** Recompute `best_ratio_recomputed = min(ratio_D1, ..., ratio_D6)` and record which denominator wins. Missing denominator families stay visible through `unresolved_family_count`.

In [ ]:
tail = lab.tail_ladder_ledger(MODE)
print("Columns used for this calculation:")
print(list(tail.columns))

display(lab.tail_ladder_explanation(MODE))
fig = lab.plot_tail_arr(MODE)
plt.show()

display(tail.head(15))

if tail["has_below_one_closure"].all():
    display(Markdown("**Result:** every tail row has a below-one closure."))
else:
    display(Markdown("**Result:** tail pressure is not silently closed. Best tested ratios stay above one, and unresolved denominator rows remain visible."))

**How to break Evidence 3A.** Find annular tail mass with no finite-overlap parent accounting and no named denominator or unresolved row. A high ratio is not automatically a break; an unnamed high ratio is.

# Evidence 3B: ARR Deficit Attribution

**Attack.** Inspect a near-miss ARR row where the current demand/capacity ratio is slightly above one.

**Ledger question.** Is the deficit arbitrary, or does a named missing channel repair it?

**Calculation.** Compare the current ARR pressure ratio with the ratio after adding renewal exposure. This is not magic; the table shows the numerator and denominator for each step.

In [ ]:
arr = lab.arr_ledger(MODE)
print("Columns used for this calculation:")
print(list(arr.columns))

display(arr)
display(lab.arr_table(MODE))

final_ratio = float(arr.iloc[-1]["ratio"])
if final_ratio < 1:
    display(Markdown("**Result:** the ARR near miss is repaired once the renewal exposure channel is named."))
else:
    display(Markdown("**Potential break:** ARR remains above one after the named correction."))

**How to break Evidence 3B.** Find a persistent ARR deficit that is not renewal exposure, boundary, tail, coherent residual, physical charge, or an explicit matrix row.

# Bonus: Coherent Residual Attribution

**Attack.** Look for a positive coherent viscous residual.

**Ledger question.** Does it become an uncharged positive source?

**Calculation.** Show signed residual balance, damping capacity, renewal/deactivation capacity, and the final status label.

In [ ]:
coh = lab.coherent_ledger(MODE)
print("Columns used for this calculation:")
print(list(coh.columns))

display(lab.coherent_explanation(MODE))
display(coh)

**How to break the coherent residual check.** Find a positive coherent residual that is not sign-balanced, damping-paid, renewal/deactivation paid, or kept as an explicit matrix row.

# Conclusion: Implications and Break Tests

The notebook is useful only if it says how to beat it. The table below is the summary claim and the attack plan.

In [ ]:
fig = lab.plot_routing_summary(MODE)
plt.show()

conclusion = lab.conclusion_summary(MODE)
display(conclusion)

## Machine-Readable Summary

The JSON output includes the scorecards and the break-test conclusion table.

In [ ]:
out = lab.write_summary_json(MODE)
print(f"wrote {lab.rel(out)}")

## CLI Equivalent

```bash
python scripts/run_lab.py --write-json
python scripts/execute_notebook.py
python scripts/check_artifacts.py
```

No-data smoke mode, not JHTDB evidence:

```bash
python scripts/run_lab.py --mode synthetic
```